# 🛡️ NetWatch — Counterfactual Cyber World Model & Ensemble Scorer
### Google Colab Cloud Runner & Live Test Harness

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sihcodesprit/external_hackathon/blob/main/NetWatch_Colab.ipynb)

This notebook lets you offload all computation, model training, test suites, and web app hosting to **Google Colab's cloud hardware** (Zero local laptop resource usage).

## 1. Initial Setup & Clone Repository
Clones your GitHub repository and installs all required dependencies + Scapy + Cloudflare tunnel.

In [ ]:
# Clone or pull latest repository
import os
if not os.path.exists('/content/external_hackathon'):
    !git clone https://github.com/sihcodesprit/external_hackathon.git /content/external_hackathon
%cd /content/external_hackathon

# Install project dependencies
!pip install -r requirements.txt
!pip install pytest scapy pycloudflared colab_ssh

print("\n✓ Environment initialized successfully in Colab!")

## 2. 🔄 Live Pull & Run Test Suite
**Run this cell whenever you make changes locally and push via `./sync_cloud.sh`.**
It instantly pulls your newest code and runs the entire test suite on Colab CPU/GPU.

In [ ]:
# Fetch latest code updates from GitHub and run pytest
!git reset --hard HEAD
!git pull origin main
!pytest tests/ -v

## 3. Train Cyber World Model & Initialize Multi-Tool Ensemble
Trains the LSTM World Model and classical ML baselines (Random Forest, Gradient Boosting, Logistic Regression) on Colab.

In [ ]:
import sys
sys.path.insert(0, '.')

from netwatch.pipeline import Pipeline

print("Training Cyber World Model and Ensemble in Colab...")
pipe = Pipeline()
pipe.load_data(n_traces=6, seed=42)
pipe.train()
pipe.evaluate()
pipe.forecast_and_simulate()
print("\n✓ Training complete! All 8 detection models are ready.")

## 4. 🌐 Launch Live Web Dashboard (Public HTTPS Tunnel)
Starts the Flask UI and creates a secure **Cloudflare Tunnel URL** so you can view and use the dashboard from your browser/phone without hosting it on your laptop.

In [ ]:
import subprocess
import time
from pycloudflared import try_cloudflare

# Start Flask app in the background
server_proc = subprocess.Popen(["python", "run.py", "--no-pipeline", "--host", "0.0.0.0", "--port", "5000"])
time.sleep(3)

# Open secure public HTTPS tunnel
tunnel = try_cloudflare(port=5000)
print("=" * 75)
print("🚀 NetWatch Web App is LIVE on Cloudflare!")
print("=" * 75)
print(f"\n👉 Dashboard Home:    {tunnel.tunnel_url}")
print(f"👉 Ensemble Scorer:   {tunnel.tunnel_url}/ensemble")
print(f"👉 PCAP Upload:       {tunnel.tunnel_url}/upload")
print(f"👉 Test Center:       {tunnel.tunnel_url}/test_center")
print("=" * 75)

## 5. 🔬 Analyze Any PCAP File via Ensemble CLI
Upload a `.pcap` or `.pcapng` file to score it across all 8 detection engines and print the consensus threat report in Colab.

In [ ]:
from google.colab import files

print("Select a .pcap / .pcapng file from your computer:")
uploaded = files.upload()

for pcap_filename in uploaded.keys():
    print(f"\nAnalyzing {pcap_filename}...")
    !python forecast_pcap.py --input "{pcap_filename}" --horizon 5

## 6. (Optional) Remote VS Code / SSH to Colab
Attach your local VS Code directly to this Colab cloud instance using `colab_ssh`.

In [ ]:
from colab_ssh import launch_ssh_cloudflared

# Set a secure password for SSH login
launch_ssh_cloudflared(password="netwatch123")